# Convert Qualtrics output to Label studio input

In [1]:
import os
import json
import csv
from datetime import datetime, timedelta, time
import spacy
import pandas as pd
import numpy as np

In [2]:
# Spacy for counting words
nlp = spacy.load("nl_core_news_sm")

In [3]:
def get_datetime(time_str):
    
    # print(type(time_str))
    
    date, time = time_str.split(' ')
    day, month, year = date.split('/')
    hours, minutes = time.split(':')

    iso_time = f'{year}-{month}-{day} {hours}:{minutes}:00'
    date_time = datetime.fromisoformat(iso_time)
    return date_time
    
    
def get_text_keys(d):
    
    text_keys = []
    for k, v in d.items():
        #k_words = [w.strip() for w in k.strip().split(' ')]
        k_words = k.strip().split(' ')
        if 'txt' in k_words or 'txt.1' in k_words:
            text_keys.append(k)
    return text_keys

def get_texts(d, text_keys):
    
    text_dict = dict()
    for tk in text_keys:
        text = d[tk]
        if type(text) == str:
            text_dict[tk] = text
        else:
            text_dict[tk] = ''
    return text_dict

def get_word_count(text_dict):
    
    total_wc = 0
    wc_dict = dict()
    for tk, text in text_dict.items():
        if str(text) != 'nan' and text != 0:
            doc = nlp(text)
            wc = len(doc)
            total_wc += wc
            wc_dict[tk] = wc
    return total_wc, wc_dict

def create_text_dict():
    
    d = {
      "id": 5,
      "data": {
        "text": ""
      },
      "annotations": [],
      "predictions": []
            }
    return d


def attention_checks(d):
    
    answers = []

        
    for k, v in d.items():
        if str(v) != 'nan':
            if ('Entitativity jong_8' in k) or ('Entitativity_oud_8' in k):
                if v == '5':
                    answers.append(True)
                else:
                    answers.append(False)
   
            if 'Man. age cat. check_2' in k:
                if v == '-2':
                    answers.append(True)
                else:
                    answers.append(False)

                
    
    if all(answers) and len(answers) > 0:
        check = True
    else:
        check = False
    return check
        

## Apply inclusion criteria

In [ ]:
path = 'DataStereotypesKim/final_dataset.xlsx'
df = pd.read_excel(path)
data = df.to_dict('records')
print(len(data[4].keys()))
print(data[4].keys())

In [ ]:
test = data[4]
c = 0
for k, v in test.items():
    if 'txt' in k:
        #print(k)
        c += 1
print(c)

In [ ]:
# get time and words

for d in data[1:]:
    
    # test = [k for k, v in d.items() if 'txt' in k]
    # print(len(test))
    text_keys = get_text_keys(d)
    #print(len(text_keys))
    text_dict = get_texts(d, text_keys)
    
    total_wc, wc_dict = get_word_count(text_dict)
    
    for k, v in wc_dict.items():
        name = f'wc_{k}'
        d[name] = v
    

    start = d['StartDate']
    end = d['EndDate']

    # start_dt = get_datetime(start)
    # end_dt = get_datetime(end)
    # duration = end_dt - start_dt
    duration = end - start
    # print(duration)
    sec = duration.total_seconds()
    minutes = sec/60

    if total_wc > 0:
        words_per_minute = total_wc/minutes
    else:
        words_per_minute = 0
    # print(words_per_minute, total_wc, minutes)
    d['duration_minutes'] = str(duration)
    d['wc'] = total_wc
    d['words_per_minute'] = round(words_per_minute, 2)


In [ ]:
# header = data[4].keys()

# path_out = 'DataStereotypesKim/final_dataset_words_per_minute.csv'

# with open(path_out, 'w') as outfile:
#     writer = csv.DictWriter(outfile, delimiter = ';', fieldnames = header)
#     writer.writeheader()
#     for d in data:
#         writer.writerow(d)


In [ ]:
df = pd.DataFrame(data)
print(len(df.columns))

In [ ]:
# for n in df.columns:
#     if 'txt' in n:
#         print(n)

In [ ]:
# mean and stdev words per minute

mean = df['words_per_minute'].mean()
std = df['words_per_minute'].std()
print(round(mean, 2), round(std, 2))

In [ ]:
# Perform all inclusion checks
  

for d in data[1:]:
    part_id = d['ParticipantID']
  
    word_counts = []
    check = attention_checks(d)

    ls_ids = []

    for k, v in d.items():
        k_words = k.split(' ')
        if (k_words[-1] == 'txt' or k_words[-1] == 'txt.1') and not k.startswith('wc_'):
            if str(v) != 'nan' and v != 0:
                text = f'{part_id} {k}\nText: {v}'
                wc_text = d[f'wc_{k}']
                if wc_text > 19:
                    word_counts.append(True)
                else:
                    word_counts.append(False)

       
              
    wc_check = all(word_counts)
    if len(word_counts) == 0:
        wc_check = False
    total_check = all([wc_check, check])
    if total_check == True:
        d['INCLUDE'] = 'TRUE'
    else:
        d['INCLUDE'] = 'FALSE'

# CSV to json 

In [ ]:
json_list = []
data_dir = 'DataStereotypesKim'
  
c = 0
for d in data[1:]:
    part_id = d['ParticipantID']
    # att_check = d['Reject: attentioncheck wrong']
    # consent = d['Consent form']
    

    if d['INCLUDE'] == 'TRUE':
        ls_ids = []

        for k, v in d.items():
            k_words = k.split(' ')
            if (k_words[-1] == 'txt' or k_words[-1] == 'txt.1') and not k.startswith('wc_'):
                if str(v) != 'nan' and v != 0:
                    text = f'{part_id} {k}\nText: {v}'
                   
                    text_d = create_text_dict()

                    # text_id = d['id']
                    text_d['id'] =  f'{part_id} {k}'
                    # text = f"Text id: {text_id}\nText: {d['text']}"
                    text_d['data']['text'] = text
                    json_list.append(text_d)
                    ls_ids.append(str(c))
                    c += 1
        d['ls-ids'] = ' '.join(ls_ids)
                
                

print(len(json_list))
    
json_str = json.dumps(json_list)

with open(f'{data_dir}/tasks.json', 'w') as outfile:
    outfile.write(json_str)

In [ ]:
df = pd.DataFrame(data)

path_out_ex = 'DataStereotypesKim/final_dataset_words_per_minute-LSID.xlsx'

df.to_excel(path_out_ex, sheet_name='Sheet1', na_rep='', float_format=None, 
            columns=None, header=True, index=False)

In [ ]:
print(len(df.columns))

## Data May 2026

In [8]:
# path = '../data/Danielle_may2026.csv'

# with open(path) as infile:
#     data = list(csv.DictReader(infile, delimiter = ','))
    

In [9]:
path = '../data/Danielle_may2026.xlsx'
df = pd.read_excel(path)

In [ ]:
create_text_dict()

In [15]:
text_dicts = []

conds = ['OV 1', 'OV 3', 'OV 3']
id_cnt = 0
for i, row in df.iterrows():
    ppn = row['ppnr']
    for c in conds:
        text = row[c]
        text_full = f'{ppn} {c}\nText: {text}'
        text_dict = create_text_dict()
        text_dict['id'] = id_cnt
        text_dict['data']['text'] = text_full
        text_dicts.append(text_dict)
        id_cnt += 1

In [17]:
text_dicts[1]

{'id': 1,
 'data': {'text': '1 OV 3\nText: Het uitgangspunt van deze mensen is te prijzen aangezien ze over het algemeens voor iets moois staan. Ze hebben wel de bedoeling om de wereld te verbeteren. Alleen voor mij wel doorgeslagen aangezien ze schade aan objecten en mensen brengen. Dit komt hun imago ook niet ten goede bij de meeste mensen, waar ik kennelijk onderdeel van blijk te zijn.'},
 'annotations': [],
 'predictions': []}

In [18]:
json_str = json.dumps(text_dicts)
path_out = '../data/Danielle_may2026.json'
with open(path_out, 'w') as outfile:
    outfile.write(json_str)

## Old conversions

In [ ]:
# Make csv
data_dir = 'Primepanel4'
files = os.listdir(data_dir)

n = 0

data = []
for file in files:
    path = f'{data_dir}/{file}'
    if file.endswith('txt'):
        
        with open(path) as infile:
            text = infile.read().strip()
        identifier = n 
        condition = file.split('.')[0]
        text_d = dict()
        text_d['id'] = n
        text_d['participant_id'] = n # to be changed
        text_d['condition'] = condition
        text_d['text'] = text
        n += 1
        data.append(text_d)

header = data[0].keys()
with open(f'{data_dir}/texts.csv', 'w') as outfile:
    writer = csv.DictWriter(outfile, fieldnames = header, delimiter = ',')
    writer.writeheader()
    for d in data:
        writer.writerow(d)


In [ ]:
# texts to json Eva

json_list = []
text_ids = []

i = 0
for text_id, text in text_dict.items():
    if text != '':
        text_with_id = f'{i}\n{text}'
        text_ids.append((text_id, i))

        text_d = create_text_dict()
        text_d['id'] =  i
        text_d['data']['text'] = text_with_id
        json_list.append(text_d)
        i+=1 
    
json_str = json.dumps(json_list)

with open(f'Eva/tasks.json', 'w') as outfile:
    outfile.write(json_str)
    

with open(f'Eva/text_ids.csv', 'w') as outfile:
    outfile.write('conditie,PPNR,text_id\n')
    for pid, text_id in text_ids:
        pn, cond = pid.split('-')
        outfile.write(f'{cond},{pn}, {text_id}\n')

In [ ]:
json_list = []

all_data = []

for f in os.listdir(data_dir):
    print(f)
    if f.endswith('csv'):
        full_path = f'{data_dir}/{f}'
        if full_path != 'Chiara4/Articles_Corriere2023.csv':
            
            print(full_path)
            with open(full_path, encoding='utf-8', errors='ignore') as infile:
                data = list(csv.DictReader(infile, delimiter = ','))
                all_data.extend(data)
print(len(all_data))
    
for d in all_data:
    print(d.keys())
    # print(d['text'])

    text_d = create_text_dict()
    text_id = d['id']
    text_d['id'] =  text_id
    text = f"Text id: {text_id}\nText: {d['text']}"
    text_d['data']['text'] = text
    json_list.append(text_d)

print(len(json_list))
    
json_str = json.dumps(json_list)

with open(f'{data_dir}/tasks.json', 'w') as outfile:
    outfile.write(json_str)



In [ ]:
#json_list